<a href="https://colab.research.google.com/github/nejumi/weave-initial-course/blob/main/notebooks/01_basics/01_traces_and_ops.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01. Traces と @weave.op

W&B Weave のトレース機能の基本から、マルチメディアコンテンツのトレースまでを学びます。

## 学習内容
1. `@weave.op` デコレータの基本
2. ネストされたトレース（コール階層）
3. マルチモーダルデータのトレース（画像・音声・動画・PDF・HTML）
4. サンプリングレート
5. カスタムコスト追跡
6. フィードバック API


In [ ]:
# ローカル環境: uv sync --all-extras で一括インストール可能
# Colab: 以下のセルで個別インストール
!pip install -q uv
!uv pip install --system -q \
  "weave>=0.51.0" \
  "wandb>=0.19.0" \
  "openai>=1.0.0" \
  "python-dotenv>=1.0.0" \
  "requests>=2.32.0" \
  "httpx>=0.27.0"

In [ ]:
import os

# Colab / ローカル環境の自動判定
try:
    import google.colab
    IN_COLAB = True
    from google.colab import userdata
    # WANDB_BASE_URL: 値がある場合のみセット（空 / 未設定 → SaaS デフォルト）
    _base_url = (userdata.get("WANDB_BASE_URL") or "").strip()
    if _base_url:
        os.environ["WANDB_BASE_URL"] = _base_url
    os.environ.setdefault("WANDB_API_KEY",  userdata.get("WANDB_API_KEY"))
    os.environ.setdefault("OPENAI_API_KEY", userdata.get("OPENAI_API_KEY"))
    os.environ.setdefault("WANDB_ENTITY",   userdata.get("WANDB_ENTITY"))
    os.environ.setdefault("WANDB_PROJECT",  userdata.get("WANDB_PROJECT") or "weave-course")
    print("Google Colab 環境")
except ImportError:
    IN_COLAB = False
    from dotenv import load_dotenv
    load_dotenv()
    # ローカル: .env に WANDB_BASE_URL= と書かれていた場合も空なら削除
    if not os.environ.get("WANDB_BASE_URL", "").strip():
        os.environ.pop("WANDB_BASE_URL", None)
    print("ローカル環境")

ENTITY  = os.environ["WANDB_ENTITY"]
PROJECT = os.environ.get("WANDB_PROJECT", "weave-course")
_target = os.environ.get("WANDB_BASE_URL", "https://api.wandb.ai (SaaS)")
print(f"Entity : {ENTITY}")
print(f"Project: {PROJECT}")
print(f"W&B URL: {_target}")


In [ ]:
import weave
client = weave.init(f"{ENTITY}/{PROJECT}")


## 1. `@weave.op` の基本

`@weave.op()` をつけるだけで、関数の入出力が自動的にトレースとして記録されます。

In [ ]:
@weave.op()
def preprocess(text: str) -> str:
    return text.strip().lower()

@weave.op()
def call_llm(prompt: str) -> str:
    from openai import OpenAI
    client_oai = OpenAI()
    resp = client_oai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=50,
    )
    return resp.choices[0].message.content

@weave.op()
def pipeline(raw_text: str) -> str:
    clean = preprocess(raw_text)
    return call_llm(f"Summarize in one sentence: {clean}")

result = pipeline("  Weights & Biases provides tools for ML practitioners.  ")
print(result)


## 2. ネストされたトレース

`@weave.op` 同士が呼び合うと、自動的に親子関係のあるトレースが生成されます。
Weave UI の **Traces** タブでコール階層をツリー表示できます。


In [ ]:
@weave.op()
def chunk_text(text: str, size: int = 100) -> list[str]:
    return [text[i:i+size] for i in range(0, len(text), size)]

@weave.op()
def embed_chunk(chunk: str) -> list[float]:
    # ダミーのエンベディング
    return [len(chunk) / 100.0, hash(chunk) % 100 / 100.0]

@weave.op()
def build_index(document: str) -> dict:
    chunks = chunk_text(document)
    embeddings = [embed_chunk(c) for c in chunks]
    return {"chunks": chunks, "embeddings": embeddings}

doc = "W&B Weave は LLM アプリケーションのオブザーバビリティと評価のためのプラットフォームです。" * 3
index = build_index(doc)
print(f"チャンク数: {len(index['chunks'])}")


## 3. マルチモーダルデータのトレース

Weave では `Annotated[bytes, Content[Literal[...]]]` 型ヒントを使うことで、
バイト列にメディアタイプを付与し、Weave UI 上でインタラクティブに可視化できます。

対応フォーマット: `png` / `jpg` / `mp3` / `mp4` / `pdf` / `html`


### 3.1 画像（OpenAI DALL-E で生成）

In [ ]:
import base64
from typing import Annotated, Literal
from weave import Content
from openai import OpenAI

@weave.op()
def generate_image(prompt: str) -> Annotated[bytes, Content[Literal["png"]]]:
    """DALL-E で画像を生成し PNG バイト列で返す"""
    oai = OpenAI()
    res = oai.images.generate(
        model="gpt-image-1",
        prompt=prompt,
        size="1024x1024",
    )
    return base64.b64decode(res.data[0].b64_json)

img_bytes = generate_image("a small robot watering a flower in a lab, pixel art style")
print(f"画像サイズ: {len(img_bytes):,} bytes")


### 3.2 音声（OpenAI TTS）

In [ ]:
import httpx

@weave.op()
def text_to_speech(text: str, voice: str = "alloy") -> Annotated[bytes, Content[Literal["mp3"]]]:
    """OpenAI TTS で音声を生成し MP3 バイト列で返す"""
    oai = OpenAI(http_client=httpx.Client(timeout=60))
    resp = oai.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice=voice,
        input=text,
        response_format="mp3",
    )
    if hasattr(resp, "read"):
        return resp.read()
    elif hasattr(resp, "content"):
        return resp.content
    return b"".join(resp.iter_bytes())

audio = text_to_speech("Weights & Biases の明確なミッションは、機械学習開発者のために最高のツールを作ることです。")
print(f"音声サイズ: {len(audio):,} bytes")


### 3.3 動画（OpenAI Sora 2）

リファレンス画像を入力として動画を生成します。


In [ ]:
import os
if os.environ.get("SKIP_EXPENSIVE"):
    print("SKIP_EXPENSIVE=1: このセルをスキップします")
else:
    import io, tempfile, requests

    REFERENCE_IMAGE_URL = "https://assets.st-note.com/img/1762403176-PimhEZu3voSeGp5C0l9t4z2N.png"

    @weave.op()
    def generate_video(
        prompt: str = "A 3D cinematic duel: a bee with a magical wand fights a dragon, dramatic lighting"
    ) -> Annotated[bytes, Content[Literal["mp4"]]]:
        """Sora 2 で動画を生成し MP4 バイト列で返す"""
        png = requests.get(REFERENCE_IMAGE_URL, timeout=30).content
        oai = OpenAI()
        v = oai.videos.create(
            model="sora-2-pro",
            prompt=prompt,
            input_reference=("ref.png", io.BytesIO(png), "image/png"),
            size="1280x720",
            seconds="12",
        )
        r = oai.videos.poll(v.id)
        if r.status != "completed":
            raise RuntimeError(f"生成失敗: {r.error}")
        b = oai.videos.download_content(v.id)
        with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as t:
            b.write_to_file(t.name)
            return open(t.name, "rb").read()

    # 生成には数分かかります
    video_bytes = generate_video()
    print(f"動画サイズ: {len(video_bytes):,} bytes")

### 3.4 PDF（arXiv 論文）

In [ ]:
import requests

@weave.op()
def fetch_arxiv_pdf(paper_id: str = "1706.03762") -> Annotated[bytes, Content[Literal["pdf"]]]:
    """arXiv の PDF をダウンロードしてそのまま返す（例: Attention Is All You Need）"""
    resp = requests.get(f"https://arxiv.org/pdf/{paper_id}.pdf", timeout=60)
    resp.raise_for_status()
    return resp.content

pdf = fetch_arxiv_pdf()
print(f"PDF サイズ: {len(pdf):,} bytes")


### 3.5 HTML

In [ ]:
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

@weave.op()
def fetch_html(url: str = "https://example.com") -> Annotated[bytes, Content[Literal["html"]]]:
    resp = requests.get(url, timeout=30, verify=False)
    resp.raise_for_status()
    return resp.text.encode("utf-8")

html = fetch_html()
print(f"HTML サイズ: {len(html):,} bytes")


## 4. サンプリングレート

高頻度に呼ばれる `op` をすべてトレースするとコストが増大します。
`tracing_sample_rate` で記録割合を制御できます。


In [ ]:
@weave.op(tracing_sample_rate=0.1)   # 約10% のみ記録
def high_freq_embedding(text: str) -> list[float]:
    return [len(text) / 100.0]

# 100 回呼んでも約 10 件しかトレースされない
for i in range(100):
    high_freq_embedding(f"sample text {i}")
print("完了（約10%のみトレース済み）")


## 5. カスタムコスト追跡

独自モデルのトークンコストを登録できます。

In [ ]:
client.add_cost(
    llm_id="my-custom-model-v1",
    prompt_token_cost=0.002,      # USD per 1K tokens
    completion_token_cost=0.006,
)
print("カスタムコスト登録完了")


## 6. フィードバック API

トレース済みの call に対して SDK/UI からフィードバックを付与できます。

In [ ]:
@weave.op()
def answer_question(question: str) -> str:
    from openai import OpenAI
    oai = OpenAI()
    resp = oai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": question}],
        max_tokens=100,
    )
    return resp.choices[0].message.content

answer, call = answer_question.call("What is W&B Weave?")
print("回答:", answer)

# フィードバックを付与
call.feedback.add_reaction("👍")
call.feedback.add_note("正確でわかりやすい回答")
call.feedback.add("quality_score", {"value": 4})
print("フィードバック付与完了")


## まとめ

次: **02_assets.ipynb** → モデル・プロンプト・データセットのアセット管理